# Indonesia Boundaries — Tier Comparison

Toggle layers in the top-right corner: **Red** = Official (`0°`), **Blue** = High-Detail (`2e-05°`), **Amber** = Simplified (`0.001°`).

In [ ]:
import json
import zipfile
from pathlib import Path

import folium
import geopandas as gpd
import shapely
from shapely.geometry import shape

REPO_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "processed-data").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not locate 'processed-data/' directory.")

DATA_DIR = REPO_ROOT / "processed-data"
OFFICIAL_ZIP = REPO_ROOT / "dist" / "official_03-districts_split-by-district.zip"


def _pairwise_overlap(geoms: list) -> float:
    return sum(
        geoms[i].intersection(geoms[j]).area
        for i in range(len(geoms))
        for j in range(i + 1, len(geoms))
    )


def compare_tiers(
    districts: list[str],
    center: tuple[float, float] | None = None,
    zoom: int = 15,
) -> folium.Map:
    detail_features, det_sizes_kb = [], {}
    for stem in districts:
        det_path = DATA_DIR / "high-detail" / "03-districts" / f"{stem}.geojson"
        det_sizes_kb[stem] = det_path.stat().st_size / 1e3
        for feat in json.loads(det_path.read_text(encoding="utf-8"))["features"]:
            detail_features.append({
                "type": "Feature",
                "properties": {**feat["properties"], "tier": "High-Detail (2e-05°)"},
                "geometry": feat["geometry"],
            })

    target_pairs = {(f["properties"].get("WADMPR"), f["properties"].get("WADMKK")) for f in detail_features}
    simp_all = json.loads((DATA_DIR / "simplified" / "03-districts" / "indonesia_districts.geojson").read_text(encoding="utf-8"))
    simplified_features = [
        {
            "type": "Feature",
            "properties": {**feat["properties"], "tier": "Simplified (0.001°)"},
            "geometry": feat["geometry"],
        }
        for feat in simp_all["features"]
        if (feat["properties"].get("WADMPR"), feat["properties"].get("WADMKK")) in target_pairs
    ]

    official_features, off_sizes_kb = [], {}
    if OFFICIAL_ZIP.is_file():
        with zipfile.ZipFile(OFFICIAL_ZIP) as zf:
            for stem in districts:
                raw_bytes = zf.read(f"03-districts/{stem}.geojson")
                off_sizes_kb[stem] = len(raw_bytes) / 1e3
                for feat in json.loads(raw_bytes.decode("utf-8"))["features"]:
                    official_features.append({
                        "type": "Feature",
                        "properties": {**feat["properties"], "tier": "Official (0°)"},
                        "geometry": feat["geometry"],
                    })
    else:
        print(f"Note: {OFFICIAL_ZIP.name} not found locally; showing High-Detail vs. Simplified only.")

    off_by_pair = {(f["properties"].get("WADMPR"), f["properties"].get("WADMKK")): shape(f["geometry"]) for f in official_features}
    det_by_pair = {(f["properties"].get("WADMPR"), f["properties"].get("WADMKK")): shape(f["geometry"]) for f in detail_features}
    simp_by_pair = {(f["properties"].get("WADMPR"), f["properties"].get("WADMKK")): shape(f["geometry"]) for f in simplified_features}

    print(f"{'District':30s} {'Off KB':>9s} {'Det KB':>9s} {'Off Pts':>9s} {'Det Pts':>9s} {'Simp Pts':>9s}")
    for stem, feat in zip(districts, detail_features):
        pair = (feat["properties"].get("WADMPR"), feat["properties"].get("WADMKK"))
        off_kb = f"{off_sizes_kb[stem]:.1f}" if stem in off_sizes_kb else "-"
        off_pts = f"{int(shapely.get_num_coordinates(off_by_pair[pair])):,d}" if pair in off_by_pair else "-"
        print(
            f"{stem:30s} {off_kb:>9s} {det_sizes_kb[stem]:9.1f} "
            f"{off_pts:>9s} "
            f"{int(shapely.get_num_coordinates(det_by_pair[pair])):9,d} "
            f"{int(shapely.get_num_coordinates(simp_by_pair[pair])):9,d}"
        )

    if len(districts) >= 2:
        parts = [f"High-Detail={_pairwise_overlap(list(det_by_pair.values())):.1f}"]
        if len(off_by_pair) >= 2:
            parts.insert(0, f"Official={_pairwise_overlap(list(off_by_pair.values())):.1f}")
        print("Pairwise overlap area -> " + ", ".join(parts))

    if center is not None:
        m = folium.Map(location=list(center), zoom_start=zoom, tiles=None, prefer_canvas=True, control_scale=True)
    else:
        m = folium.Map(tiles=None, prefer_canvas=True, control_scale=True)

    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}",
        attr="Esri",
        name="Esri Light Gray",
        max_zoom=16,
        show=True,
    ).add_to(m)
    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Dark_Gray_Base/MapServer/tile/{z}/{y}/{x}",
        attr="Esri",
        name="Esri Dark Gray",
        max_zoom=16,
        show=False,
    ).add_to(m)
    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Topo_Map/MapServer/tile/{z}/{y}/{x}",
        attr="Esri",
        name="Esri World Topo",
        show=False,
    ).add_to(m)
    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
        attr="Esri",
        name="Esri World Imagery",
        show=False,
    ).add_to(m)

    def _add_layer(features_list: list[dict], label: str, color: str, weight: float, dash: str | None) -> None:
        props = features_list[0]["properties"] if features_list else {}
        fields = [k for k in ("WADMKK", "tier") if k in props]
        style = {"color": color, "fillColor": color, "weight": weight, "fillOpacity": 0.0}
        if dash:
            style["dashArray"] = dash
        folium.GeoJson(
            data={"type": "FeatureCollection", "features": features_list},
            name=label,
            show=True,
            style_function=lambda _feat, s=style: dict(s),
            tooltip=folium.GeoJsonTooltip(fields=fields, sticky=True) if fields else None,
        ).add_to(m)

    if official_features:
        _add_layer(official_features, "Official (0°)", "#dc2626", 2.5, None)
    _add_layer(detail_features, "High-Detail (2e-05°)", "#2563eb", 2.0, "6, 4")
    _add_layer(simplified_features, "Simplified (0.001°)", "#d97706", 2.0, "2, 4")

    if center is None:
        minx, miny, maxx, maxy = gpd.GeoDataFrame.from_features(detail_features, crs="EPSG:4326").total_bounds
        m.fit_bounds([[float(miny), float(minx)], [float(maxy), float(maxx)]])

    folium.LayerControl(collapsed=False).add_to(m)
    return m

In [ ]:
# Inland border junction (Jalan Raya Boyolali-Kartosuro / Simpang Tiga Ngasem)
compare_tiers(
    ["Kota_Surakarta", "Boyolali", "Sukoharjo", "Karanganyar"],
    center=(-7.541561, 110.713238),
    zoom=16,
)

In [ ]:
# Small islands (switch basemap to Esri World Imagery to inspect tiny islets)
compare_tiers(["Administrasi_Kepulauan_Seribu"])